# Task 8a — recursion at scale, across models: the Act 8 test on seven models

**Run all cells.** Everything below is idempotent and resumable: re-running a cell is always safe, and the
scoring phase picks up where it left off after a runtime disconnect. One runtime scores **one model**
(`MODEL_KEY`); the record is assembled from whichever per-model caches exist, so the seven models can be
scored on seven runtimes and merged at the end.

**The question.** After an embedded clause closes (`The soldier that the singer trusted |`), does the model's
next-token state sit closer to the bare subject's (`The soldier |`) than a control of the same token length that
ends on the same word but has no clause to close (`The singer heard the soldier trusted |`)? Locally the
answer was yes on GPT-2 (1.43×) and Qwen2.5-0.5B (1.12×). This run scores the same test on seven models with a
**200-frame item set identical across all of them**, so that whatever differs between models is the model.

**The measure.** At each prefix the state is `log_softmax(logits[-1])` in float32; the distance between two
states is the L2 norm of their difference. Per (embedded E, control C) pair: `ret = ‖s_E − s_REF‖`,
`ctrl = ‖s_C − s_REF‖`. The record holds the raw distances, the verb mass at every stop, `log P(V)` at every
stop and the ten top next tokens after every pass; the ratios, intervals and tests are computed from it outside
this repository.

**The pre-registered reading** (pass = every base model's depth-1 ratio intervals clear 1.0 with the
token-matched control) is applied to the record, not here. **Nothing in the record is read before the anchor
check passes**: Qwen2.5-0.5B must reproduce the local run's 30 frames to 2 % (`ret`, `ctrl`) and 0.01 (verb
mass), or the pipeline differs from the report's and this notebook stops.

| phase | what | where |
|---|---|---|
| 0 | load the seven **tokenizers** only, generate the frames once (seed 8), save `frames_8a_seed8.json` | any runtime |
| anchor | score `data/frames_8a_local30.json` on Qwen2.5-0.5B and compare | the `qwen25_0.5b` runtime |
| A | score one model's 14 passes × 200 frames into `scores_8a_<MODEL_KEY>.jsonl` (resumable) | one runtime per model |
| B | merge frames + caches + anchor into `record_8a_multimodel.json` | any runtime, CPU |

## 1 · Bootstrap — clone the repo

In [ ]:
#@title Clone (or update) the repository { display-mode: "form" }
import base64, json, os, subprocess, sys, textwrap, urllib.error, urllib.request
from pathlib import Path

REPO_OWNER  = "SalmonSung"
REPO_NAME   = "m1_llms_analyzer"
REPO_BRANCH = "main"   #@param {type:"string"}

CLEAN_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
NEW_PAT_URL = "https://github.com/settings/personal-access-tokens/new"


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _colab_secret(name):
    """Read a Colab secret, returning None if it is absent or access is denied."""
    if not _in_colab():
        return None
    try:
        from google.colab import userdata
        return userdata.get(name) or None
    except Exception as exc:
        print(f"  (Colab secret {name!r} unavailable: {type(exc).__name__})")
        return None


def _clean(token):
    """Strip whitespace and stray quotes -- by far the most common paste error."""
    if not token:
        return None
    return token.strip().strip('"').strip("'").strip() or None


def _token_kind(token):
    """Name the token type from its prefix, without revealing the value."""
    for prefix, kind in (
        ("github_pat_", "fine-grained PAT"),
        ("ghp_", "classic PAT"),
        ("gho_", "OAuth token"),
        ("ghs_", "App installation token"),
        ("ghu_", "user-to-server token"),
    ):
        if token.startswith(prefix):
            return kind
    return "UNRECOGNISED PREFIX"


def _api(path, token):
    """GET api.github.com/<path> with the token. Raises urllib.error.HTTPError."""
    request = urllib.request.Request(
        f"https://api.github.com/{path}",
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode())


def _auth_config(token):
    """Auth as a per-command git config value, so it never touches .git/config or a URL."""
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    return f"http.extraHeader=AUTHORIZATION: basic {basic}"


def _run(cmd, token=None):
    """Run git, redacting the auth header and the token from anything printed."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr or result.stdout
        if token:
            message = message.replace(token, "***")
        shown = " ".join("<auth>" if "extraHeader" in c else c for c in cmd)
        raise RuntimeError(f"git failed: {shown}\n{message}")
    return result.stdout.strip()


def _preflight(token):
    """Verify the token before git runs, so failures name their actual cause.

    A bare `git clone` failure says only 'Invalid username or token', which covers an
    expired token, a typo, a missing repo grant, and un-authorised SSO alike. These two
    API calls tell those apart.
    """
    kind = _token_kind(token)
    print(f"GITHUB_TOKEN: {len(token)} chars, looks like a {kind}.")
    if kind == "UNRECOGNISED PREFIX":
        print("  Warning: GitHub tokens start with github_pat_, ghp_, gho_, ghs_ or ghu_.")
        print("  If you pasted an account password or an SSH key, that will not work here.")

    try:
        me = _api("user", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 401:
            raise SystemExit(textwrap.dedent(f"""
                GitHub rejected this token (401 Unauthorized). The token itself is bad --
                this is not a permissions problem. Most likely one of:

                  * it has expired (fine-grained PATs expire, 30 days by default);
                  * it was revoked or regenerated;
                  * the secret holds something that is not a token (an account password
                    will never work -- GitHub removed password auth for git);
                  * it was truncated or mangled when pasted.

                Fix: create a new token at
                  {NEW_PAT_URL}
                  - Resource owner: {REPO_OWNER}
                  - Repository access: only select repositories -> {REPO_NAME}
                  - Permissions: Repository permissions -> Contents -> Read-only
                Then in Colab: key icon in the left sidebar -> edit GITHUB_TOKEN, paste the
                new value with no quotes and no trailing spaces, keep 'Notebook access' on,
                and re-run this cell.
            """).strip())
        if exc.code == 403:
            raise SystemExit(textwrap.dedent(f"""
                GitHub returned 403 for this token. Usually either a rate limit, or the
                token needs SAML SSO authorisation for the '{REPO_OWNER}' organisation.
                If {REPO_OWNER} is an org with SSO, open your token's settings page and
                click 'Configure SSO' -> Authorize.

                Original error: {exc}
            """).strip())
        raise

    print(f"  Authenticates as: {me.get('login')}")

    try:
        repo = _api(f"repos/{REPO_OWNER}/{REPO_NAME}", token)
    except urllib.error.HTTPError as exc:
        if exc.code == 404:
            login = me.get("login")
            not_owner = (
                f"\n                  * you are {login}, but the repo belongs to "
                f"{REPO_OWNER} and you are not a collaborator on it;"
                if login and login.lower() != REPO_OWNER.lower()
                else ""
            )
            raise SystemExit(textwrap.dedent(f"""
                The token is valid (you are {login}), but it cannot see
                {REPO_OWNER}/{REPO_NAME}. GitHub returns 404 rather than 403 for a private
                repo a token has no grant on, so this means one of:

                  * the token's 'Repository access' does not include {REPO_NAME};
                  * it lacks the 'Contents: Read-only' repository permission;{not_owner}
                  * the owner or name is misspelled (both are case-sensitive).

                Fix: open {NEW_PAT_URL} (or edit the existing token), grant this
                repository and Contents: Read-only, then re-run this cell.
            """).strip())
        raise

    print(f"  Repo access:      OK ({'private' if repo.get('private') else 'public'})")
    return True


_TOKEN = _clean(_colab_secret("GITHUB_TOKEN") or os.environ.get("GITHUB_TOKEN"))

if not _in_colab() and Path("pyproject.toml").exists():
    # Running from a local checkout -- nothing to clone.
    REPO_DIR = Path.cwd()
    print(f"Local checkout detected: {REPO_DIR}")
else:
    REPO_DIR = Path("/content") / REPO_NAME if _in_colab() else Path.cwd() / REPO_NAME
    if not _TOKEN:
        raise SystemExit(textwrap.dedent(f"""
            GITHUB_TOKEN is not set, and {REPO_OWNER}/{REPO_NAME} is private.
              1. Create a fine-grained PAT at {NEW_PAT_URL}
                 - Resource owner: {REPO_OWNER}
                 - Repository access: only select repositories -> {REPO_NAME}
                 - Permissions: Contents -> Read-only
              2. Colab left sidebar -> key icon -> add a secret named GITHUB_TOKEN
              3. Turn on 'Notebook access' for it, then re-run this cell.
        """).strip())

    _preflight(_TOKEN)

    # Auth travels as a per-command header, never in the URL. Nothing is written to
    # .git/config, so there is no token left on disk to scrub afterwards.
    _AUTH = _auth_config(_TOKEN)
    try:
        if REPO_DIR.exists():
            print(f"\nRepo already present at {REPO_DIR}; updating...")
            _run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", CLEAN_URL], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "-c", _AUTH, "fetch", "origin", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], _TOKEN)
            _run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_BRANCH}"], _TOKEN)
        else:
            print(f"\nCloning into {REPO_DIR} ...")
            _run(["git", "-c", _AUTH, "clone", "--branch", REPO_BRANCH, "--depth", "1",
                  CLEAN_URL, str(REPO_DIR)], _TOKEN)
    except RuntimeError as exc:
        # The API accepted the token but git did not -- rare, and worth naming, because
        # the obvious readings (bad token, missing grant) were just ruled out above.
        raise SystemExit(textwrap.dedent(f"""
            {exc}

            The token passed the API preflight above, so it is valid and can see this
            repo -- the failure is in the git transport itself. Things to check:

              * branch '{REPO_BRANCH}' exists on the remote (a typo in REPO_BRANCH gives
                'Remote branch not found');
              * a stale {REPO_DIR} from an earlier run: delete it and re-run this cell;
              * a corporate proxy or VPN intercepting HTTPS to github.com.
        """).strip())
    print("Done. Remote is", CLEAN_URL, "(no credentials stored on disk).")

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
print("Commit:", _run(["git", "rev-parse", "--short", "HEAD"]))
del _TOKEN  # do not leave the token bound in the notebook namespace

## 2 · Dependencies

Installed with `--upgrade-strategy only-if-needed` so Colab's preinstalled,
CUDA-matched `torch` is **kept** rather than reinstalled (a torch swap costs several
minutes and can break GPU support).

In [ ]:
#@title Install dependencies
import subprocess, sys

print("Installing (quiet; ~30s on a cold runtime)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--upgrade-strategy", "only-if-needed", "-e", "."],
    capture_output=True, text=True,
)
print(result.stdout[-2000:] or "(no output)")
if result.returncode != 0:
    print(result.stderr[-3000:], file=sys.stderr)
    raise SystemExit("Dependency installation failed -- see the error above.")

# Make the freshly installed package importable in this already-running kernel.
import importlib, site
importlib.reload(site)
for module in [m for m in list(sys.modules) if m.startswith("m1_analyzer")]:
    del sys.modules[module]

import m1_analyzer
print("m1_analyzer", m1_analyzer.__version__, "ready")

## 3 · Environment report

In [ ]:
#@title What am I running on?
import torch, transformers, numpy, platform
from m1_analyzer import in_colab, resolve_hf_token

print(f"python        : {platform.python_version()}")
print(f"torch         : {torch.__version__}")
print(f"transformers  : {transformers.__version__}")
print(f"numpy         : {numpy.__version__}")
print(f"in Colab      : {in_colab()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU           : {props.name} ({props.total_memory / 1024**3:.1f} GB)")
    print(f"CUDA          : {torch.version.cuda}")
else:
    print("GPU           : none -- running on CPU.")
    print("                Runtime -> Change runtime type -> T4 GPU for anything above ~1B params.")

# Only reports presence. The token value is never printed.
print(f"HF_TOKEN      : {'found' if resolve_hf_token() else 'not set (fine for ungated models)'}")

# The dtype the scorer will run in. A T4 gets float16 (its bfloat16 is emulated and slow);
# Ampere and newer get bfloat16; CPU gets float32.
from m1_analyzer.utils.device import resolve_device, resolve_dtype
print(f"scoring dtype : {str(resolve_dtype(resolve_device('auto'), 'auto')).replace('torch.', '')}")

## 4 · Configuration

**This is the only cell you normally edit.** Pick `MODEL_KEY` for this runtime. The seven keys, their default
Hugging Face ids and the anchor are in `experiments/task_8a.py::MODELS_8A`; `HF_ID_OVERRIDES` /
`REVISION_OVERRIDES` (JSON) replace an id or pin a commit without editing the package — the record stores what
was actually loaded. `DEVICE_MAP="auto"` streams a 20–60 GB checkpoint straight onto the GPU (gpt-oss-20b,
Gemma-4-31B: an 80 GB runtime). The frame generator's `SEED` / `N_FRAMES` / `MAX_DRAWS` are the
pre-registration; they are shown, not tuned, and the frames file pins them.

In [ ]:
#@title Run configuration { display-mode: "form" }
MODEL_KEY        = "qwen25_0.5b"  #@param ["qwen25_0.5b", "qwen3_0.6b", "qwen3_1.7b", "qwen3_8b", "llama31_8b", "gptoss_20b", "gemma4_31b", "all-small"]
HF_ID_OVERRIDES  = "{}"           #@param {type:"string"}
REVISION_OVERRIDES = "{}"         #@param {type:"string"}
TRUST_REMOTE_CODE = False         #@param {type:"boolean"}
DTYPE            = "auto"         #@param ["auto", "bfloat16", "float16", "float32"]
DEVICE_MAP       = ""             #@param ["", "auto"]
FRAMES_PER_BATCH = 8              #@param {type:"integer"}
# --- the pre-registered item set (shown, not tuned) ---
N_FRAMES         = 200            #@param {type:"integer"}
SEED             = 8              #@param {type:"integer"}
MAX_DRAWS        = 3000           #@param {type:"integer"}
FRAMES_FILE      = "frames_8a_seed8.json"        #@param {type:"string"}
# --- the anchor ---
ANCHOR_FILE      = "data/frames_8a_local30.json" #@param {type:"string"}
TOL_REL          = 0.02           #@param {type:"number"}
TOL_VERBMASS     = 0.01           #@param {type:"number"}
# --- run mechanics ---
LIMIT            = 0              #@param {type:"integer"}
RESUME           = True           #@param {type:"boolean"}
MIRROR_EVERY     = 10             #@param {type:"integer"}
MIRROR_TO_DRIVE  = False          #@param {type:"boolean"}
DRIVE_DIR        = "/content/drive/MyDrive/m1_llms_analyzer/task_8a"  #@param {type:"string"}
OUTPUT_DIR       = "outputs/task_8a"             #@param {type:"string"}

import json
import shutil
from pathlib import Path

from m1_analyzer import ModelConfig, RunConfig, ScoringConfig, StorageConfig, in_colab
from m1_analyzer.experiments import ANCHOR_KEY, CODES_8A, MODELS_8A, SMALL_KEYS

OUT = Path(OUTPUT_DIR)
OUT.mkdir(parents=True, exist_ok=True)
KEYS = list(SMALL_KEYS) if MODEL_KEY == "all-small" else [MODEL_KEY]
RUN_TAG = f"8a_{MODEL_KEY}"

_hf_ids = json.loads(HF_ID_OVERRIDES or "{}")
_revisions = json.loads(REVISION_OVERRIDES or "{}")
SPECS = {
    key: spec.with_overrides(
        hf_id=_hf_ids.get(key, spec.hf_id), revision=_revisions.get(key, spec.revision),
        trust_remote_code=bool(TRUST_REMOTE_CODE or spec.trust_remote_code),
    )
    for key, spec in MODELS_8A.items()
}

FRAMES_PATH = OUT / FRAMES_FILE                    # phase 0: the shared item set
ANCHOR_PATH = Path(ANCHOR_FILE)                    # read-only input (the local run's 30 frames)
ANCHOR_OUT = OUT / "anchor_8a.json"                # the anchor comparison, in full
RECORD_PATH = OUT / "record_8a_multimodel.json"    # the deliverable (phase B)


def cache_for(key: str) -> Path:
    """Phase A cache of one model: one JSONL line per frame, all fourteen passes."""
    return OUT / f"scores_8a_{key}.jsonl"


def make_config(spec) -> RunConfig:
    """LM head, no extra BOS: the prefix ids already carry whatever the tokenizer adds by default."""
    return RunConfig(
        model=ModelConfig(model_id=spec.hf_id, revision=spec.revision, head="causal_lm", dtype=DTYPE,
                          trust_remote_code=spec.trust_remote_code, device_map=DEVICE_MAP or None),
        scoring=ScoringConfig(bos_policy="none", batch_size=len(CODES_8A) * FRAMES_PER_BATCH, max_length_cap=512),
        storage=StorageConfig(output_dir=str(OUT)),
        seed=SEED,
    )


def drive_dir() -> Path | None:
    """The Drive folder, mounted on first use; None outside Colab or when mirroring is off."""
    if not (MIRROR_TO_DRIVE and in_colab()):
        return None
    target = Path(DRIVE_DIR)
    if not Path("/content/drive").exists():
        from google.colab import drive  # noqa: F401 -- Colab only
        drive.mount("/content/drive")
    target.mkdir(parents=True, exist_ok=True)
    return target


def restore(path: Path) -> bool:
    """Copy `path` back from Drive when it is missing locally. True if it is now present."""
    if path.exists():
        return True
    drive = drive_dir()
    if drive is not None and (drive / path.name).exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(drive / path.name, path)
        print(f"restored {path.name} from {drive}")
    return path.exists()


def mirror_path(path: Path):
    drive = drive_dir()
    return None if drive is None else drive / path.name


print(f"models this runtime : {KEYS}")
for key in KEYS:
    s = SPECS[key]
    print(f"  {key:<12} {s.hf_id}  rev={s.revision or 'default'}  {'(gated)' if s.gated else ''}{'' if s.base else '(post-trained: outside the pass rule)'}")
print(f"frames              : {FRAMES_PATH}  (seed {SEED}, {N_FRAMES} frames, {MAX_DRAWS} draws max)")
print(f"anchor              : {ANCHOR_PATH} -> {ANCHOR_OUT}")
print(f"record              : {RECORD_PATH}")
print("Configuration ready.")

## 5 · Smoke test (no download, a few seconds)

Runs the *entire* chain on a random 4-layer GPT-2 whose word-level vocabulary spells every frame word as one
token: three frames generated with its tokenizer, scored, cached, resumed, a synthetic anchor compared, the
record built and validated. If this passes, the only things that can still go wrong with a real model are the
download, GPU memory, and the Drive paths. The numbers are meaningless.

In [ ]:
#@title Smoke test on a tiny model
import json
import tempfile
import time

from m1_analyzer import Analyzer
from m1_analyzer.experiments import (
    FRAME_WORDS_8A, ModelSpec, anchor_check, build_record_8a, generate_frames_8a, rows_from_frame, save_frames_8a,
    score_model_8a, validate_record_8a,
)
from m1_analyzer.testing import build_tiny_local_model

started = time.perf_counter()
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    tiny_path = build_tiny_local_model(tmp / "tiny", extra_vocab=FRAME_WORDS_8A, max_positions=64)
    smoke = Analyzer(RunConfig(model=ModelConfig(model_id=tiny_path, head="causal_lm"),
                               scoring=ScoringConfig(bos_policy="none", batch_size=64)))
    spec = ModelSpec("tiny", tiny_path)
    frames = save_frames_8a(generate_frames_8a({"tiny": smoke.models.tokenizer}, n=3, seed=SEED), tmp / "frames.json")
    header, rows = score_model_8a(smoke, spec, frames, cache_path=tmp / "scores.jsonl", show_progress=False)
    _, again = score_model_8a(smoke, spec, frames, cache_path=tmp / "scores.jsonl", show_progress=False)
    assert again == rows, "resume rescored something"
    # A synthetic anchor from our own numbers must pass exactly.
    reference = []
    for row in rows:
        for r in rows_from_frame(row):
            reference.append({**r, "stop_tok": row["last_tok"][r["structure"]], "n_tok": row["n_tok"][r["structure"]]})
    anchor_file = tmp / "anchor.json"
    anchor_file.write_text(json.dumps({"seed": SEED, "n_frames": 3, "frames": json.load(open(frames))["frames"],
                                       "qwen25_reference_rows": reference}))
    block = anchor_check(smoke, spec, anchor_file, tol_rel=TOL_REL, tol_verbmass=TOL_VERBMASS)
    assert block["pass"], block
    record = build_record_8a(frames, {"tiny": tmp / "scores.jsonl"}, anchor=block, allow_partial=True)
    validate_record_8a(record)
    smoke.unload()
n = len(record["frames"])
print(f"{n} frames, {len(record['models']['tiny']['rows'])} rows, {len(record['models']['tiny']['tokens'])} token entries, "
      f"{len(record['models']['tiny']['top10'])} top-10 lists; anchor pass={record['anchor']['pass']}; "
      f"{time.perf_counter() - started:.1f}s")
print("\nSmoke test passed (the numbers are meaningless: a random 5 MB model).")

## 6 · Phase 0 — the shared frames (tokenizers only)

The frame list is generated **once**, with seed 8, and kept only where it passes every assertion in **all
seven** tokenizers: single-token pool words with a leading space, equal token counts per (embedded, control)
pair under each tokenizer's default `add_special_tokens`, equal last token for the `_B` controls. So all seven
tokenizers are loaded here (a few MB each; Llama-3.1 and Gemma-4 are gated — accept their licences and set an
`HF_TOKEN` Colab secret). A frames file already on disk or on Drive is reused as is, so every runtime scores the
identical list; it is refused if it was not filtered on all seven.

In [ ]:
#@title Generate (or restore) the frames
import time

from m1_analyzer import resolve_hf_token
from m1_analyzer.experiments import generate_frames_8a, load_frames_8a, load_tokenizers, save_frames_8a
from m1_analyzer.experiments.jsonl_cache import mirror

ALL_KEYS = list(MODELS_8A)
if not RESUME and FRAMES_PATH.exists():
    FRAMES_PATH.unlink()
if restore(FRAMES_PATH):
    frames_payload = load_frames_8a(FRAMES_PATH)
    print(f"Using the existing frames file {FRAMES_PATH} ({frames_payload['meta']['n_frames']} frames).")
else:
    started = time.perf_counter()
    tokenizers = load_tokenizers(ALL_KEYS, hf_ids={k: s.hf_id for k, s in SPECS.items()}, token=resolve_hf_token(),
                                 trust_remote_code={k: s.trust_remote_code for k, s in SPECS.items()})
    print(f"{len(tokenizers)} tokenizers loaded in {time.perf_counter() - started:.0f}s")
    frames_payload = generate_frames_8a(tokenizers, n=N_FRAMES, seed=SEED, max_draws=MAX_DRAWS)
    save_frames_8a(frames_payload, FRAMES_PATH)
    if mirror_path(FRAMES_PATH):
        mirror(FRAMES_PATH, mirror_path(FRAMES_PATH))
    del tokenizers

meta = frames_payload["meta"]
if set(meta["tokenizers_filtered_on"]) != set(ALL_KEYS):
    raise SystemExit(f"{FRAMES_PATH} was filtered on {meta['tokenizers_filtered_on']}, not all seven models. "
                     "Delete it (or set RESUME=False) and re-run this cell.")
print(f"seed {meta['seed']}: {meta['n_frames']} / {meta['n_frames_requested']} frames from {meta['pools']['draws']} draws; "
      f"pools after the seven-tokenizer filter: {meta['pools']}")
if meta["n_frames"] < meta["n_frames_requested"]:
    print(f"NOTE: only {meta['n_frames']} frames passed within {MAX_DRAWS} draws; the record says so in meta.")
f0 = frames_payload["frames"][0]
print("\nframe 0:")
for code in CODES_8A:
    print(f"  {code:<7} {f0['sentences'][code]}")

## 7 · Load the model

The first model of `KEYS`, **with its language-model head** and `bos_policy="none"`: the prefix ids are
`tok(prefix)` exactly as the tokenizer returns them (a BOS for Llama / Gemma, none for Qwen / gpt-oss — the
record's `bos_added`), and the state is read at the last id. The dtype the weights load in is recorded; the
logits are cast to float32 before the log-softmax regardless.

In [ ]:
#@title Load the model
import time

from m1_analyzer import Analyzer
from m1_analyzer.experiments import model_meta_8a, verb_pool_kept

CURRENT_KEY = KEYS[0]
started = time.perf_counter()
analyzer = Analyzer(make_config(SPECS[CURRENT_KEY]))
print(f"Loaded {CURRENT_KEY} in {time.perf_counter() - started:.1f}s\n")
for key, value in analyzer.describe().items():
    if key != "layers":
        print(f"{key:>18} : {value}")
meta = model_meta_8a(analyzer, SPECS[CURRENT_KEY])
kept = verb_pool_kept(analyzer.models.tokenizer)
print(f"{'bos_added':>18} : {meta['bos_added']}")
print(f"{'weight dtype':>18} : {meta['weight_dtype']}  (logits in float32)")
print(f"{'parameters':>18} : {meta['n_params_b']} B")
print(f"{'vocabulary':>18} : {meta['vocab_size']} (the dimension every distance lives in)")
print(f"{'verb pool kept':>18} : {len(kept)} / 70 single-token verbs"
      + ("" if len(kept) == 70 else f"; dropped: {sorted(set(frames_payload['verb_pool']) - set(kept))}"))

## 8 · Anchor — reproduce the local record first

Only on the `qwen25_0.5b` runtime. The 30 local frames and Qwen2.5-0.5B's `ret`, `ctrl` and verb masses for
all 210 rows are read from `ANCHOR_FILE` (never written). Before any forward pass, our token counts and last
tokens are compared with the rows' `n_tok` / `stop_tok`; then the 30 frames are scored with this pipeline and
every row must satisfy `|ret − ours| / ours < 0.02`, likewise `ctrl`, and `|verbmass − ours| < 0.01`. The full
comparison goes to `anchor_8a.json`; the record keeps its summary. **A failure stops the notebook** and names
what most likely differs (BOS handling, weight dtype, a trailing space) — the pipeline is then not the report's
and the seven-model numbers would not be comparable.

In [ ]:
#@title Anchor check (qwen25_0.5b only)
import json
import time

from m1_analyzer.experiments import anchor_check, anchor_diagnosis

if CURRENT_KEY != ANCHOR_KEY:
    print(f"{CURRENT_KEY} is not the anchor model; the anchor is checked on the {ANCHOR_KEY} runtime.")
elif not ANCHOR_PATH.exists():
    print(f"WARNING: no anchor file at {ANCHOR_PATH}. The pipeline is NOT verified against the local record; "
          "put frames_8a_local30.json there (it is committed under data/) and re-run this cell before trusting the numbers.")
else:
    started = time.perf_counter()
    block = anchor_check(analyzer, SPECS[ANCHOR_KEY], ANCHOR_PATH, tol_rel=TOL_REL, tol_verbmass=TOL_VERBMASS,
                         frames_per_batch=FRAMES_PER_BATCH, show_progress=True)
    ANCHOR_OUT.write_text(json.dumps(block, indent=1), encoding="utf-8")
    if mirror_path(ANCHOR_OUT):
        shutil.copy(ANCHOR_OUT, mirror_path(ANCHOR_OUT))
    print(f"\n{block['n_rows']} rows compared in {time.perf_counter() - started:.1f}s "
          f"(hf_id {block['hf_id']}, revision {str(block['revision'])[:12]}, weights {block['weight_dtype']}, bos_added {block['bos_added']})")
    print(f"  max rel diff ret      : {block['max_rel_diff_ret']:.5f}   (tol {TOL_REL})")
    print(f"  max rel diff ctrl     : {block['max_rel_diff_ctrl']:.5f}   (tol {TOL_REL})")
    print(f"  max abs diff verbmass : {block['max_abs_diff_verbmass']:.5f}   (tol {TOL_VERBMASS})")
    print(f"  token mismatches      : {len(block['token_mismatches'])}")
    print(f"  PASS                  : {block['pass']}")
    if not block["pass"]:
        print()
        print(anchor_diagnosis(block, analyzer=analyzer, anchor_path=ANCHOR_PATH))
        print("\nIf the probes do not reproduce the local value and the excess is small, re-run with DTYPE=float32 "
              "(the local run was a 0.5B model; float16 on a T4 can move an L2 distance by a few percent).")
        raise SystemExit("Anchor failed: the pipeline differs from the local run. Do not score the 200 frames.")

## 9 · One frame, end to end

Frame 0 on the loaded model: the fourteen prefixes, their token counts and last tokens (the assertions the
filter enforced), the distance of every pass to `REF`, the verb mass at every stop and the top guesses after the
embedded and control passes. This is what one cache line holds.

In [ ]:
#@title Walk through one frame
from m1_analyzer.experiments import PAIRS_8A, rows_from_frame, score_frames_batch
from m1_analyzer.experiments.task_8a import verb_ids

FRAME_INDEX = 0  #@param {type:"integer"}

frame = frames_payload["frames"][FRAME_INDEX]
_, ids = verb_ids(analyzer.models.tokenizer)
[row] = score_frames_batch(analyzer.states, analyzer.models.tokenizer, [(FRAME_INDEX, frame)], verb_token_ids=ids)
print(f"frame {FRAME_INDEX}: N1={frame['N1']} V={frame['V']} ({row['seconds']:.2f}s for 14 passes)\n")
print(f"{'code':<7} {'n_tok':>5}  {'last_tok':<12} {'dist_to_REF':>11}  {'verbmass':>8}  {'logP(V)':>8}  top-3")
for code in CODES_8A:
    top = ", ".join(f"{t!r} {p:.2f}" for t, p in row["top10"][code][:3])
    d = "-" if code == "REF" else f"{row['dist_to_ref'][code]:.2f}"
    print(f"{code:<7} {row['n_tok'][code]:>5}  {row['last_tok'][code]!r:<12} {d:>11}  {row['verbmass'][code]:>8.4f}  {row['logp_V'][code]:>8.3f}  {top}")
print()
for r in rows_from_frame(row):
    print(f"  {r['structure']:<5} vs {r['control']:<6}  ret {r['ret']:8.2f}  ctrl {r['ctrl']:8.2f}  ctrl/ret {r['ctrl'] / r['ret'] if r['ret'] else float('nan'):.3f}")

## 10 · Phase A — score every frame (GPU, resumable)

For each model of `KEYS` (one, or the three small ones): `FRAMES_PER_BATCH` frames × 14 prefixes per forward
batch, the scalars computed on the CPU, one fsynced JSONL line per frame. The **header is the
pre-registration**: model key, `hf_id`, resolved revision, weight dtype, `bos_added`, vocabulary, the frames
file's sha256, the kept verb pool and every measure's definition; a resume under a different model or frames
file refuses and names the field. On the anchor runtime this cell refuses to score the 200 frames unless the
anchor passed. Lower `FRAMES_PER_BATCH` on an out-of-memory error (the batch also halves itself).

In [ ]:
#@title Score all frames for every model of this runtime (resumable)
import json
import time

from m1_analyzer import Analyzer
from m1_analyzer.experiments import score_model_8a

PROVENANCE = {"notebook": "experiment_8a.ipynb", "run_tag": RUN_TAG, "device_map": DEVICE_MAP or None, "dtype_setting": DTYPE}
for key in KEYS:
    spec = SPECS[key]
    cache = cache_for(key)
    if key == ANCHOR_KEY and ANCHOR_PATH.exists():
        if not ANCHOR_OUT.exists() or not json.loads(ANCHOR_OUT.read_text(encoding="utf-8")).get("pass"):
            raise SystemExit("Run and pass the anchor check (section 8) before scoring the 200 frames on qwen25_0.5b.")
    if not RESUME and cache.exists():
        cache.unlink()
    restore(cache)
    if key != CURRENT_KEY:
        analyzer.unload()
        started = time.perf_counter()
        analyzer = Analyzer(make_config(spec))
        CURRENT_KEY = key
        print(f"Loaded {key} in {time.perf_counter() - started:.1f}s")
    started = time.perf_counter()
    header, rows = score_model_8a(
        analyzer, spec, FRAMES_PATH, cache_path=cache, frames_per_batch=FRAMES_PER_BATCH, provenance=PROVENANCE,
        show_progress=True, mirror_path=mirror_path(cache), mirror_every=MIRROR_EVERY, limit=LIMIT or None,
    )
    print(f"{key}: {len(rows)} / {header['n_frames']} frames in {cache} ({time.perf_counter() - started:.0f}s this session; "
          f"weights {header['weight_dtype']}, revision {str(header['revision'])[:12]}, bos_added {header['bos_added']})")
    if LIMIT:
        print(f"  dry run: LIMIT={LIMIT}; set LIMIT=0 and re-run to score everything.")

## 11 · Phase B — the record (CPU, from whichever caches exist)

Gathers `scores_8a_<key>.jsonl` for all seven keys from local disk or Drive, checks each against the frames
file (sha256) and for completeness, and assembles `record_8a_multimodel.json`: `meta`, `verb_pool`, `frames`,
`models{key: meta, verb_pool_kept, tokens, rows, top10}`, `anchor`. Models without a cache are listed as
missing; run their runtimes and re-run this cell. The per-structure medians printed are **descriptive only**
— the ratios, intervals and tests are computed from the record outside the repo.

In [ ]:
#@title Assemble the record from every cache present
import json
import statistics

from m1_analyzer.experiments import PAIRS_8A, build_record_8a, missing_models

caches = {}
for key in MODELS_8A:
    if restore(cache_for(key)):
        caches[key] = cache_for(key)
missing = missing_models(caches)
print(f"caches found : {sorted(caches)}")
print(f"missing      : {missing or 'none'}")
restore(ANCHOR_OUT)
anchor_block = json.loads(ANCHOR_OUT.read_text(encoding="utf-8")) if ANCHOR_OUT.exists() else None
if anchor_block is None:
    print("WARNING: no anchor_8a.json; the record's anchor block will be null.")
elif not anchor_block["pass"]:
    print("WARNING: the recorded anchor check FAILED; the record carries pass=false and should not be read.")

record = build_record_8a(FRAMES_PATH, caches, anchor=anchor_block, allow_partial=True)
n = record["meta"]["n_frames"]
print(f"\n{n} frames; {len(record['models'])} model(s) in the record"
      + (f"; anchor pass={record['anchor']['pass']}" if record["anchor"] else ""))
print(f"\n{'model':<12} {'weights':<9} {'bos':<5} {'rows':>5} {'kept':>5}   median ctrl/ret per structure (descriptive only)")
for key, block in record["models"].items():
    meds = []
    for e, c in PAIRS_8A:
        rs = [r for r in block["rows"] if r["structure"] == e and r["control"] == c and r["ret"] > 0]
        meds.append(f"{e}/{c} {statistics.median(r['ctrl'] / r['ret'] for r in rs):.2f}")
    m = block["meta"]
    print(f"{key:<12} {m['weight_dtype']:<9} {str(m['bos_added']):<5} {len(block['rows']):>5} {len(block['verb_pool_kept']):>5}   " + "  ".join(meds))

## 12 · Save and persist

The record is the experiment's single deliverable (one JSON, raw rows, no summaries). Colab wipes `/content`
when the runtime is recycled, so the Drive cell copies the frames file, every cache, the anchor comparison and
the record to `DRIVE_DIR`. The anchor input under `data/` is never written.

In [ ]:
#@title Save the record
import json
import os
import tempfile

def atomic_write_json(path, payload):
    path = Path(path)
    fd, tmp = tempfile.mkstemp(dir=str(path.parent), prefix=f".{path.stem}.", suffix=".json")
    os.close(fd)
    Path(tmp).write_text(json.dumps(payload, indent=1, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)

atomic_write_json(RECORD_PATH, record)
print(f"record : {RECORD_PATH} ({RECORD_PATH.stat().st_size / 1024 / 1024:.1f} MB, "
      f"{len(record['frames'])} frames, models {list(record['models'])})")
print(json.dumps({"meta": record["meta"], "anchor": record["anchor"],
                  "models": {k: v["meta"] for k, v in record["models"].items()}}, indent=2))

In [ ]:
#@title Persist results to Google Drive (survives a runtime disconnect)
import shutil

from m1_analyzer import in_colab

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    target = Path(DRIVE_DIR)
    target.mkdir(parents=True, exist_ok=True)
    for path in [FRAMES_PATH, ANCHOR_OUT, RECORD_PATH] + [cache_for(k) for k in MODELS_8A]:
        if path.exists():
            shutil.copy(path, target / path.name)
            print("copied", target / path.name)
else:
    print("Not running in Colab -- skipping Drive mount.")

In [ ]:
#@title Download the record to your machine
from m1_analyzer import in_colab

if in_colab():
    from google.colab import files
    if RECORD_PATH.exists():
        files.download(str(RECORD_PATH))
else:
    print("Not in Colab; the record is already on disk under", OUT)


In [ ]:
#@title Run the test suite inside Colab
import subprocess, sys

result_proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True
)
print(result_proc.stdout[-4000:])
print(result_proc.stderr[-2000:], file=sys.stderr)

---

## Troubleshooting

| Symptom | Fix |
|---|---|
| `Access to 'meta-llama/...' was denied` / same for Gemma in phase 0 | Accept the licence on the model page, create a read token, add it as the `HF_TOKEN` Colab secret with notebook access, re-run. All seven tokenizers are needed before any frame is drawn. |
| `Model 'google/gemma-4-31b' was not found` | The Gemma-4 id is a best guess: put the base checkpoint's exact id in `HF_ID_OVERRIDES` as `{"gemma4_31b": "<org/name>"}`; the record stores what was loaded. |
| CUDA out of memory while loading gpt-oss-20b / Gemma-4-31B | Use an 80 GB runtime (A100 80 GB / H100) and `DEVICE_MAP="auto"`; the checkpoint then streams to the GPU. gpt-oss dequantises to bf16 (~42 GB) on GPUs without MXFP4 kernels. |
| CUDA out of memory during scoring | Lower `FRAMES_PER_BATCH` (the batch also halves itself on OOM); a 262k-vocabulary model's logits for 112 short prefixes are ~1.5 GB. |
| Anchor fails with `token mismatch` on every row | A BOS is being added that the local run did not have (or vice versa): the prefix ids must be `tok(prefix)` with the tokenizer's default; `ScoringConfig(bos_policy="none")` must stay. |
| Anchor fails by a few percent, probes do not reproduce it | Weight dtype: re-run with `DTYPE="float32"` (0.5B fits everywhere). If it still fails, stop and report; do not score the 200 frames. |
| `was written for frames_sha256=...` on resume | The frames file changed since the cache was started. Restore the original frames file from Drive, or delete the cache to rescore. |
| `was filtered on [...], not all seven models` | A frames file from a partial run is on disk; delete it (or `RESUME=False`) so phase 0 regenerates it with all seven tokenizers. |
| Phase B lists models as missing | Their caches are on other runtimes' disks: copy them to `DRIVE_DIR` (section 12 of each runtime) and set `MIRROR_TO_DRIVE=True` here so phase B restores them. |
| Only N < 200 frames passed | The seven-tokenizer intersection was too small for 3,000 draws; `meta.n_frames` says how many, and the record is delivered as is. |